# Beta e custo do capital próprio da empresa do seu grupo
**ED0139 · Finanças Corporativas · 2026-2 · Prof. Sérgio Cardoso · UFC**

Versão do aluno. Entrega pelo SIGAA, no prazo publicado no ambiente.

Este notebook refaz, em Python, exatamente a mesma conta da planilha. A regra da disciplina
é que os dois caminhos cheguem ao mesmo número: método diferente com resultado igual é a
defesa contra erro de fórmula e contra copiar sem entender.

**Esta tarefa é mais curta que a anterior.** Tudo o que você já fez na tarefa de risco e
retorno vem pronto aqui: ler os preços, corrigir o grupamento, calcular retornos e desenhar o
gráfico. Você só preenche **o que é novo**, que são três células:

1. o beta da sua empresa contra o BOVA11, pelos **três caminhos** que a planilha usa;
2. a decomposição do risco total em parte de mercado e parte própria, e o R²;
3. o custo do capital próprio pelo CAPM, com as premissas declaradas da disciplina.

Depois vem a leitura, que é o que de fato se avalia: o que esse beta diz sobre o negócio da
sua empresa.

Os dados são do pacote congelado da disciplina: fechamentos oficiais da B3 entre
20/08/2021 e 20/08/2026, sem ajuste por proventos.

---

## Antes de começar, leia isto

Este notebook roda **na nuvem do Google**. Você não instala nada no seu computador.

**1. Salve uma cópia sua, agora.** Menu **Arquivo → Salvar uma cópia no Drive**.
Se você pular este passo, tudo o que escrever se perde ao fechar a aba.

**2. Para executar uma célula**, clique no botão ▶ que aparece à esquerda dela,
ou aperte **Shift + Enter**.

**3. Execute na ordem, de cima para baixo.** Uma célula pulada quebra todas as de
baixo, porque cada uma usa o resultado da anterior.

**4. Você só escreve onde estiver marcado** `>>> SUA VEZ`. O resto já está pronto e
não deve ser alterado.

**5. Se parar de responder ou você ficar muito tempo longe**, o Colab desconecta.
Não perdeu nada: use **Ambiente de execução → Executar tudo** e ele refaz tudo em
alguns segundos.

**6. Não precisa baixar nem enviar arquivo de dados.** Os preços são lidos direto
da internet.

---

## Passo 0 · Identificação

Preencha os três campos abaixo e execute a célula. Eles entram no arquivo que você
vai entregar.

In [ ]:
#@title Identificação { display-mode: "form" }
NOME = ""       #@param {type:"string"}
MATRICULA = ""  #@param {type:"string"}
GRUPO = ""      #@param {type:"string"}

print(f"{NOME or '(sem nome)'} · matrícula {MATRICULA or '(vazia)'} · grupo {GRUPO or '(vazio)'}")

## Passo 1 · Os dados e as premissas (pronto)

Os preços são lidos direto do pacote congelado da disciplina, pela internet. Duas coisas
mudam de um grupo para outro: o ticker e, só para a Hapvida, o fator do grupamento.

As duas premissas do CAPM são **declaradas**, não estimadas: taxa sem risco de 14,00% ao ano
(Selic meta, série 432 do Banco Central, em 20/08/2026) e prêmio de mercado esperado de
4,23% ao ano (prêmio de mercado maduro em Damodaran, 05/01/2026). Não use o prêmio total do
Brasil de 7,47% sobre a Selic: ele embute 3,24% de risco país, que a Selic já remunera.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CAMINHO = "https://sergiocardoso.pro.br/ufc/financas-corporativas/2026-2/dados/precos/fechamentos_diarios_2021-2026_congelado_2026-08-24.csv"
TICKER  = "ABEV3"   # <<< o ticker do SEU grupo
INDICE  = "BOVA11"
PREGOES_ANO = 252

RF = 0.14          # taxa sem risco ao ano
PREMIO = 0.0423     # prêmio de mercado ESPERADO ao ano

precos = pd.read_csv(CAMINHO, parse_dates=["data"]).set_index("data")
precos[[TICKER, INDICE]].tail()

**A Hapvida fez grupamento de 15 para 1 em 06/06/2025**, e os preços anteriores a essa data
precisam ser multiplicados por 15 antes de qualquer estatística. É a mesma correção da tarefa
anterior, então ela **vem pronta**: a célula abaixo já escolhe o fator pelo ticker. Execute e
siga em frente.

In [ ]:
DATA_EVENTO = "2025-06-06"
FATOR = 15 if TICKER == "HAPV3" else 1     # só a Hapvida teve evento no período

serie = precos[TICKER].copy()
if FATOR != 1:
    serie.loc[:DATA_EVENTO] = serie.loc[:DATA_EVENTO] * FATOR
    serie.loc[DATA_EVENTO] = precos[TICKER].loc[DATA_EVENTO]   # o próprio dia ex já vem agrupado

serie.tail(3)

## Passo 2 · Os dois retornos, alinhados (pronto)

O beta compara a ação com o índice **dia a dia**. Os dois retornos precisam estar na mesma
frequência e nas mesmas datas: calcular o da ação em base diária e o do índice em base
semanal é um dos erros mais comuns desta tarefa. O outro é rodar a regressão sobre os
**preços** em vez dos retornos, o que produz um número grande e sem significado.

A célula abaixo **vem pronta** e já resolve os dois: leia o comentário nela antes de executar.

In [ ]:
# concat + dropna alinha as duas séries pelas MESMAS datas. É isso que impede
# o erro de comparar a ação num dia com o índice em outro.
par = pd.concat([serie.pct_change(), precos[INDICE].pct_change()], axis=1).dropna()
par.columns = [TICKER, INDICE]

ra = par[TICKER]      # retorno da ação
rm = par[INDICE]      # retorno do índice

print(f"{len(par)} pares de retorno, de {par.index.min():%d/%m/%Y} a {par.index.max():%d/%m/%Y}")
par.head(3)

## Passo 3 · O beta pelos três caminhos

$$\beta_i = \frac{\mathrm{Cov}(R_i, R_M)}{\sigma^2(R_M)}
\qquad
\beta_i = \rho_{iM}\,\frac{\sigma_i}{\sigma_M}
\qquad
\beta_i = \text{inclinação da reta de } R_i \text{ contra } R_M$$

São a mesma conta escrita de três formas. Os três **têm de dar o mesmo número**, e é isso
que a conferência mais abaixo verifica.

Repare que a anualização não aparece aqui: como o 252 multiplica em cima e embaixo, ele se
cancela e o beta é o mesmo em base diária ou anual.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

# caminho 1: covariância sobre variância
cov = ...          # dica: par.cov(ddof=1).iloc[0, 1]
var_m = ...        # dica: a variância dos retornos do índice, com ddof=1
beta_1 = ...

# caminho 2: correlação vezes a razão dos riscos
corr = ...         # dica: par.corr().iloc[0, 1]
sd_a = ...         # dica: o desvio dos retornos da AÇÃO, com ddof=1
sd_m = ...         # dica: o desvio dos retornos do ÍNDICE, com ddof=1
beta_2 = ...

# caminho 3: a inclinação da reta
# dica: np.polyfit(x, y, 1) devolve DOIS valores, a inclinação e o intercepto, nessa ordem.
#       Aqui o x é o retorno do índice e o y é o retorno da ação. Não troque a ordem:
#       inverter x e y dá outro número, e ele não é o beta.
beta_3, alfa_dia = ...

print(f"caminho 1 (cov / var) ......... {beta_1:.4f}")
print(f"caminho 2 (corr x razão) ...... {beta_2:.4f}")
print(f"caminho 3 (inclinação) ........ {beta_3:.4f}")
print(f"maior diferença entre os três . {max(beta_1, beta_2, beta_3) - min(beta_1, beta_2, beta_3):.10f}")

In [ ]:
beta = beta_1
assert max(beta_1, beta_2, beta_3) - min(beta_1, beta_2, beta_3) < 1e-9, \
    "os três caminhos discordam: revise qual deles está com a série trocada"
print(f"beta de {TICKER}: {beta:.4f}")

## Passo 4 · A decomposição do risco

$$\underbrace{\sigma_i^2}_{\text{risco total}}
= \underbrace{\beta_i^2\,\sigma_M^2}_{\text{de mercado}}
+ \underbrace{\sigma^2(\varepsilon_i)}_{\text{da empresa}}
\qquad\qquad
R^2 = \frac{\beta_i^2\,\sigma_M^2}{\sigma_i^2} = \rho_{iM}^2$$

A soma vale para **variâncias**, nunca para desvios-padrão. Somar as duas partes em
desvio-padrão é o terceiro erro clássico da tarefa, e ele infla o risco total.

Aqui a anualização volta a importar: variância multiplica por 252 e desvio multiplica pela
**raiz** de 252.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

risco_total = ...       # dica: o desvio DIÁRIO da ação vezes a raiz de PREGOES_ANO
risco_indice = ...      # dica: o mesmo, para o índice
risco_mercado = ...     # dica: o módulo do beta vezes o risco do índice -> abs(beta) * ...
risco_proprio = ...     # dica: a raiz de (risco_total ao quadrado MENOS risco_mercado ao quadrado)
                        #       ao quadrado em Python: x**2
r2 = ...                # dica: a correlação ao quadrado

print(f"risco total da ação ....... {risco_total:.2%}")
print(f"risco do índice ........... {risco_indice:.2%}")
print(f"parte de mercado .......... {risco_mercado:.2%}")
print(f"parte própria ............. {risco_proprio:.2%}")
print(f"R² (fração de mercado) .... {r2:.4f}")
print(f"confere: {risco_mercado**2 + risco_proprio**2:.10f} contra {risco_total**2:.10f}")

## Passo 5 · O custo do capital próprio

$$E(R_i) = R_F + \beta_i\,\bigl[\,E(R_M) - R_F\,\bigr]$$

O único dado específico da sua empresa que entra nesta conta é o **beta**. Tamanho, setor e
endividamento não aparecem aqui, e isso é ao mesmo tempo a força e a fraqueza do modelo.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

ke = ...     # dica: a taxa sem risco MAIS o beta vezes o prêmio

print(f"Ke de {TICKER} = {RF:.2%} + {beta:.4f} × {PREMIO:.2%} = {ke:.2%}")

### Confira contra a planilha

Escreva abaixo os dois números que a **sua planilha** produziu. Se o beta discordar na
quarta casa, quase sempre é intervalo errado na planilha: uma linha a mais ou a menos numa
das duas colunas de retorno.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

EXCEL_BETA = ...    # copie da sua planilha
EXCEL_KE   = ...

print(f"beta: Python {beta:.4f} × Excel {EXCEL_BETA:.4f}")
print(f"Ke:   Python {ke:.2%} × Excel {EXCEL_KE:.2%}")

## Passo 6 · O desenho (pronto)

Cada ponto é um pregão: no eixo horizontal o que o índice fez naquele dia, no vertical o que
a sua empresa fez. A reta é o beta. A distância dos pontos até a reta é o risco próprio, o
pedaço que a carteira apaga.

O gráfico **vem pronto**: desenhar em matplotlib não ensina finanças, e o que interessa aqui é
olhar o resultado. Execute e compare com as nuvens que você viu em sala.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.scatter(rm * 100, ra * 100, s=6, alpha=.35, color="#005386")
x = np.linspace(rm.min(), rm.max(), 50)
ax.plot(x * 100, (alfa_dia + beta_3 * x) * 100, color="#E77813", linewidth=2.2)
ax.axhline(0, color="#C9D2D9", linewidth=.8)
ax.axvline(0, color="#C9D2D9", linewidth=.8)
ax.set_xlabel(f"retorno diário do {INDICE} (%)")
ax.set_ylabel(f"retorno diário de {TICKER} (%)")
ax.set_title(f"Linha característica · beta {beta:.2f} · R² {r2:.2f}", loc="left", fontsize=12)
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## Passo 7 · A leitura

Responda em texto, no máximo cinco linhas cada, na célula abaixo.

1. O beta da sua empresa ficou acima ou abaixo de 1? O que no negócio dela explica isso?
2. O R² ficou alto ou baixo? O que significa um R² baixo com risco total alto?
3. Descontar dez anos de fluxo ao Ke da sua empresa, em vez de ao Ke da empresa de menor
   beta da turma, muda o valor presente para mais ou para menos? Por quê?

*(escreva aqui)*

1.

2.

3.

## Se der erro

Os erros que quase sempre acontecem, e o que fazer:

**`NameError: name 'beta' is not defined`**
Você pulou uma célula, ou a sessão caiu. Faça **Ambiente de execução → Executar tudo**
e depois volte de onde parou.

**`TypeError: unsupported operand type(s) for ...: 'ellipsis'`**
Sobrou um `...` que você não substituiu. O `...` é o buraco a preencher, não é código.
Procure na célula que deu erro.

**`AssertionError: os três caminhos discordam`**
Um dos três está com a série trocada. O suspeito mais comum é o `np.polyfit`, com o x e o y
invertidos: o índice vai no x e a ação no y.

**A célula não faz nada e fica com um círculo girando**
O Colab está reconectando. Espere alguns segundos. Se não voltar, faça
**Ambiente de execução → Reiniciar sessão** e depois **Executar tudo**.

Nenhum desses erros apaga o seu trabalho. O que você escreveu continua escrito.

## Passo 8 · O seu comentário

Escreva, em poucas linhas, o que os números da sua empresa dizem. Não é resumo do que
você fez: é leitura do resultado. Cite números que só o seu ticker produz.

Escreva **entre as três aspas**, sem apagá-las.

In [ ]:
COMENTARIO = """
(escreva aqui)

Sugestões do que responder, se travar:
- o beta ficou acima ou abaixo de 1, e isso combina com o que a empresa vende?
- quanto do risco dela é mercado e quanto é história própria?
- o Ke ficou em quanto, e o que isso significa para quem for avaliar essa empresa?
- Python e Excel bateram? se não, onde estava a diferença?
"""

print(f"{len(COMENTARIO.split())} palavras escritas")

## O que entregar

**Dois arquivos, pelo SIGAA, no prazo publicado no ambiente:** o arquivo `.md` que a célula
abaixo gera e a **planilha preenchida**.

Os números dos dois têm de bater. Erro honesto e documentado vale mais que resultado
certo sem rastro: o arquivo registra o que você calculou, inclusive o que ficou em
branco, e é assim que a correção enxerga onde a conta parou.

O que se avalia não é o número: é a **leitura**. O que o beta da sua empresa diz sobre o
negócio dela, e o que o R² diz sobre quanto dessa história é mercado.

*Dados: pacote congelado da disciplina. Fonte primária dos preços: B3. Taxa sem risco:
Banco Central, SGS 432. Prêmio esperado: Damodaran, 05/01/2026.*

In [ ]:
#@title Gerar o arquivo de entrega (execute por último)
import re, json, unicodedata
from datetime import datetime

def _g(nome):
    """Lê a variável pelo nome. Devolve None se ela nem chegou a existir."""
    return globals().get(nome, None)

def _v(x):
    """Devolve o número, ou None se o aluno não chegou a calcular."""
    try:
        if x is Ellipsis:
            return None
        return float(x)
    except Exception:
        return None

def _pct(x, casas=2):
    v = _v(x)
    return "não calculado" if v is None else f"{v*100:.{casas}f}%".replace(".", ",")

def _num(x, casas=6):
    v = _v(x)
    return "não calculado" if v is None else f"{v:.{casas}f}".replace(".", ",")

def _dif(a, b, casas=4):
    va, vb = _v(a), _v(b)
    if va is None or vb is None:
        return "não dá para comparar"
    return f"{va - vb:+.{casas}f}".replace(".", ",")

def _difpp(a, b):
    va, vb = _v(a), _v(b)
    if va is None or vb is None:
        return "não dá para comparar"
    return f"{(va - vb) * 100:+.3f}".replace(".", ",") + " p.p."

# a data de congelamento vem do próprio nome do arquivo lido
_m = re.search(r"congelado_(\d{4})-(\d{2})-(\d{2})", _g("CAMINHO") or "")
BASE_EM = f"{_m.group(3)}/{_m.group(2)}/{_m.group(1)}" if _m else "não identificada"

_b1, _b2, _b3 = _v(_g("beta_1")), _v(_g("beta_2")), _v(_g("beta_3"))
_tres = [x for x in (_b1, _b2, _b3) if x is not None]
_amp = f"{max(_tres) - min(_tres):.10f}".replace(".", ",") if len(_tres) == 3 else "não dá para comparar"

_linhas = [
    f"# Beta e custo do capital próprio · {_g('TICKER') or '?'}",
    "",
    "**ED0139 · Finanças Corporativas · 2026-2 · UFC**",
    "",
    f"- **Aluno:** {_g('NOME') or '(não preenchido)'}",
    f"- **Matrícula:** {_g('MATRICULA') or '(não preenchida)'}",
    f"- **Grupo:** {_g('GRUPO') or '(não preenchido)'}",
    f"- **Empresa:** {_g('TICKER') or '(não definida)'}",
    f"- **Índice de referência:** {_g('INDICE') or '(não definido)'}",
    f"- **Base congelada em:** {BASE_EM}",
    f"- **Premissas:** taxa sem risco {_pct(_g('RF'))} · prêmio esperado {_pct(_g('PREMIO'))}",
    f"- **Gerado em:** {datetime.now():%d/%m/%Y %H:%M}",
    "",
    "## O beta pelos três caminhos",
    "",
    "| caminho | beta |",
    "|---|---|",
    f"| 1 · covariância sobre variância | {_num(_g('beta_1'), 4)} |",
    f"| 2 · correlação vezes a razão dos riscos | {_num(_g('beta_2'), 4)} |",
    f"| 3 · inclinação da reta | {_num(_g('beta_3'), 4)} |",
    f"| maior diferença entre os três | {_amp} |",
    "",
    "## Risco e custo de capital",
    "",
    "| medida | valor |",
    "|---|---|",
    f"| covariância diária com o índice | {_num(_g('cov'), 8)} |",
    f"| correlação com o índice | {_num(_g('corr'), 4)} |",
    f"| risco total da ação, ao ano | {_pct(_g('risco_total'))} |",
    f"| risco do índice, ao ano | {_pct(_g('risco_indice'))} |",
    f"| parte de mercado, ao ano | {_pct(_g('risco_mercado'))} |",
    f"| parte própria, ao ano | {_pct(_g('risco_proprio'))} |",
    f"| R² | {_num(_g('r2'), 4)} |",
    f"| custo do capital próprio (Ke) | {_pct(_g('ke'))} |",
    "",
    "## Python × Excel",
    "",
    "| medida | Python | Excel | diferença |",
    "|---|---|---|---|",
    f"| beta | {_num(_g('beta'), 4)} | {_num(_g('EXCEL_BETA'), 4)} | {_dif(_g('beta'), _g('EXCEL_BETA'))} |",
    f"| custo do capital próprio | {_pct(_g('ke'))} | {_pct(_g('EXCEL_KE'))} | {_difpp(_g('ke'), _g('EXCEL_KE'))} |",
    "",
    "## Comentário",
    "",
    (_g("COMENTARIO") or "").strip() or "(não escrito)",
    "",
    "---",
    "",
    "```json",
    json.dumps({
        "ticker": _g("TICKER"), "indice": _g("INDICE"), "matricula": _g("MATRICULA"),
        "grupo": _g("GRUPO"), "base_congelada_em": BASE_EM, "fonte": _g("CAMINHO"),
        "pares_de_retorno": int(len(_g("par"))) if _g("par") is not None else None,
        "rf": _v(_g("RF")), "premio": _v(_g("PREMIO")),
        "beta_1": _b1, "beta_2": _b2, "beta_3": _b3,
        "cov_dia": _v(_g("cov")), "corr": _v(_g("corr")),
        "risco_total": _v(_g("risco_total")), "risco_indice": _v(_g("risco_indice")),
        "risco_mercado": _v(_g("risco_mercado")), "risco_proprio": _v(_g("risco_proprio")),
        "r2": _v(_g("r2")), "ke": _v(_g("ke")),
        "excel_beta": _v(_g("EXCEL_BETA")), "excel_ke": _v(_g("EXCEL_KE")),
    }, ensure_ascii=False, indent=2),
    "```",
]
_texto = "\n".join(_linhas)

_id = unicodedata.normalize("NFKD", f"{_g('MATRICULA') or _g('NOME') or 'sem_identificacao'}")
_id = re.sub(r"[^A-Za-z0-9]+", "_", _id).strip("_")[:40] or "sem_identificacao"
ARQUIVO = f"FinCorp_06_{_g('TICKER') or 'sem_ticker'}_{_id}.md"

with open(ARQUIVO, "w", encoding="utf-8") as f:
    f.write(_texto)

_faltando = [c for c in ["NOME", "MATRICULA", "GRUPO"] if not _g(c)]
if _faltando:
    print("ATENÇÃO: falta preencher " + ", ".join(_faltando) + " no Passo 0.\n")
if "não calculado" in _texto:
    print("ATENÇÃO: há medidas não calculadas. O arquivo registra isso, e tudo bem;\n"
          "         mas confira se você não pulou uma célula sem querer.\n")

print(f"Arquivo gerado: {ARQUIVO}\n")
print("-" * 62)
print(_texto[:900])
print("-" * 62)

try:
    from google.colab import files
    files.download(ARQUIVO)
    print("\nO download começou. Se o navegador bloquear, use o painel de arquivos\n"
          "à esquerda (ícone de pasta), clique nos três pontos do arquivo e baixe.")
except Exception:
    print(f"\nO arquivo foi salvo ao lado deste notebook, com o nome acima.")